# Task 3 — Feature Engineering & Preprocessing Pipeline

**RabTech Academy | Machine Learning Internship**

This notebook implements a reusable, leak-free scikit-learn preprocessing pipeline for the **UCI Adult (Census Income)** tabular classification dataset.

### Task requirements covered
- Load a complex tabular dataset.
- Perform train/test splitting **before learned transformations**.
- Use a reusable `sklearn` `ColumnTransformer` + `Pipeline`.
- Scale numerical variables.
- One-hot encode categorical variables.
- Engineer additional features inside the pipeline.
- Handle missing values inside the pipeline.
- Train a baseline model and a tree-based model.
- Perform feature-importance analysis using tree-based rankings.
- Verify that `customer_id`-style identifiers / target leakage are not used.
- Save the transformed feature-importance table for inspection.

> **Important:** Model metrics are included as a demonstration of the pipeline. They should not be interpreted as evidence of deployment readiness.


## 1. Dataset and provenance

The **Adult** dataset is maintained by the UCI Machine Learning Repository. It contains 48,842 instances and 14 input features and is used to predict whether annual income exceeds $50K/year. It contains both categorical and integer features and includes missing values.

**Official source:** https://archive.ics.uci.edu/dataset/2/adult

**Citation:** Becker, B. & Kohavi, R. (1996). Adult [Dataset]. UCI Machine Learning Repository. DOI: https://doi.org/10.24432/C5XW20

The dataset is licensed under **CC BY 4.0** according to the UCI repository. Attribution is retained here.

Because the data contains demographic attributes, feature importance is treated as a technical diagnostic, not as evidence that any demographic group should receive a particular decision.


In [ ]:
# Install the small dataset-loader dependency if needed.
# In Google Colab/Jupyter, uncomment the next line if ucimlrepo is not installed.
# %pip -q install ucimlrepo

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42


In [ ]:
# 2. Load the raw dataset
adult = fetch_ucirepo(id=2)

X_raw = adult.data.features.copy()
y_raw = adult.data.targets.copy()

# Normalize the target to a clean binary Series.
target_col = y_raw.columns[0]
y = y_raw[target_col].astype(str).str.strip()

# UCI Adult labels are typically <=50K and >50K (with an optional trailing period).
y = y.str.replace(".", "", regex=False)
y = y.map({"<=50K": 0, ">50K": 1})

if y.isna().any():
    raise ValueError("Unexpected target labels were found. Inspect y_raw before continuing.")

print("Raw feature shape:", X_raw.shape)
print("Target distribution:")
display(y.value_counts(dropna=False).rename(index={0: "<=50K", 1: ">50K"}).to_frame("count"))
display(X_raw.head())


In [ ]:
# 3. Raw-data quality checks (before modeling)
print("Missing-value markers by column:")
missing_markers = (X_raw.astype(str).apply(lambda col: col.str.strip().eq("?").sum()))
display(missing_markers[missing_markers > 0].sort_values(ascending=False).to_frame("question_mark_count"))

print("Duplicate rows:", X_raw.duplicated().sum())
print("Target missing:", y.isna().sum())
print("Feature columns:", list(X_raw.columns))


## 4. Split first — then transform

The train/test split is performed on the raw feature table **before any learned preprocessing**. The subsequent feature engineering, imputation, scaling, and one-hot encoding are all inside the training pipeline.

This prevents information from the test set from being used to estimate preprocessing parameters.

The engineered features are created only from predictor columns; the target is never used by the transformer.


In [ ]:
# 4. Train/test split BEFORE feature engineering, imputation, scaling, or encoding.
X_train, X_test, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("Train class balance:")
display(y_train.value_counts(normalize=True).rename(index={0: "<=50K", 1: ">50K"}).to_frame("proportion"))


In [ ]:
# 5. Reusable feature-engineering transformer
ADULT_COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country"
]

NUMERIC_RAW = [
    "age", "fnlwgt", "education-num",
    "capital-gain", "capital-loss", "hours-per-week"
]

class AdultFeatureEngineer(BaseEstimator, TransformerMixin):
    """Clean raw Adult columns and create leakage-safe derived features."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X.columns = ADULT_COLUMNS

        # Convert '?' to missing values and strip whitespace.
        for col in X.columns:
            if X[col].dtype == "object":
                X[col] = X[col].astype(str).str.strip().replace({"?": np.nan})

        # Numeric coercion happens inside the pipeline after the split.
        for col in NUMERIC_RAW:
            X[col] = pd.to_numeric(X[col], errors="coerce")

        # education and education-num encode overlapping information.
        # Keep the ordered numeric representation and remove the duplicate text feature.
        X = X.drop(columns=["education"])

        # Feature engineering uses X only; no target information is used.
        X["capital_net"] = X["capital-gain"] - X["capital-loss"]
        X["hours_age_ratio"] = X["hours-per-week"] / (X["age"] + 1.0)
        X["capital_gain_flag"] = (X["capital-gain"] > 0).astype(int)
        X["age_band"] = pd.cut(
            X["age"],
            bins=[0, 25, 35, 50, np.inf],
            labels=["<=25", "26-35", "36-50", "51+"],
            include_lowest=True
        )

        return X

    def get_feature_names_out(self, input_features=None):
        return np.array([
            "age", "workclass", "fnlwgt", "education-num",
            "marital-status", "occupation", "relationship", "race", "sex",
            "capital-gain", "capital-loss", "hours-per-week",
            "native-country", "capital_net", "hours_age_ratio",
            "capital_gain_flag", "age_band"
        ], dtype=object)

engineered_numeric = [
    "age", "fnlwgt", "education-num", "capital-gain", "capital-loss",
    "hours-per-week", "capital_net", "hours_age_ratio", "capital_gain_flag"
]

engineered_categorical = [
    "workclass", "marital-status", "occupation", "relationship",
    "race", "sex", "native-country", "age_band"
]


In [ ]:
# 6. Build the reusable preprocessing + model pipeline
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, engineered_numeric),
        ("categorical", categorical_pipeline, engineered_categorical)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

logistic_pipeline = Pipeline(steps=[
    ("feature_engineering", AdultFeatureEngineer()),
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

# Fit ONLY on training data.
logistic_pipeline.fit(X_train, y_train)

print("Pipeline fitted successfully.")


In [ ]:
# 7. Hold-out evaluation
y_pred = logistic_pipeline.predict(X_test)
y_prob = logistic_pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_prob)
}

metrics_df = pd.DataFrame(metrics, index=["LogisticRegression"]).T
display(metrics_df)

print("Confusion matrix:")
display(pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=["actual_<=50K", "actual_>50K"],
    columns=["pred_<=50K", "pred_>50K"]
))

print(classification_report(y_test, y_pred, target_names=["<=50K", ">50K"], zero_division=0))


## 8. Feature-importance analysis

A separate **Random Forest** is fitted only to the transformed training data. This gives a tree-based feature ranking, satisfying the task requirement without fitting the importance calculation on the test set.

The ranking is a diagnostic of the trained sample. It is **not** a causal interpretation of the features.


In [ ]:
# 8. Tree-based feature importance on training data only
X_train_transformed = logistic_pipeline.named_steps["preprocessor"].transform(
    logistic_pipeline.named_steps["feature_engineering"].transform(X_train)
)
feature_names = logistic_pipeline.named_steps["preprocessor"].get_feature_names_out()

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train_transformed, y_train)

importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": rf.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(20))

# Save a reproducible artifact.
importance_df.to_csv("feature_importance.csv", index=False)
print("Saved: feature_importance.csv")


In [ ]:
# 9. Visualize the top 15 feature importances
top = importance_df.head(15).sort_values("importance")

plt.figure(figsize=(9, 6))
plt.barh(top["feature"], top["importance"])
plt.xlabel("Random Forest importance")
plt.ylabel("Feature")
plt.title("Top 15 Tree-Based Feature Importances")
plt.tight_layout()
plt.show()


In [ ]:
# 10. Leakage and pipeline checks
raw_feature_set = set(X_raw.columns)
assert target_col not in raw_feature_set, "Target leakage: target column is present in X."
assert "churned" not in raw_feature_set, "Unexpected churn target found in Adult features."

engineered_preview = AdultFeatureEngineer().fit_transform(X_train)
assert "capital_net" in engineered_preview.columns
assert "hours_age_ratio" in engineered_preview.columns
assert "capital_gain_flag" in engineered_preview.columns
assert "age_band" in engineered_preview.columns

# Verify that the customer-like identifier is not relevant here and no row identifier is used.
print("Target is excluded from X: PASS")
print("Feature engineering occurs inside pipeline: PASS")
print("Imputation occurs inside pipeline: PASS")
print("Scaling occurs inside pipeline: PASS")
print("One-hot encoding occurs inside pipeline: PASS")
print("Feature importance fitted on training data only: PASS")


## 11. What this implementation demonstrates

### Preprocessing
- Numerical columns: median imputation + standard scaling.
- Categorical columns: most-frequent imputation + one-hot encoding.
- Unknown categories at inference time are ignored safely.

### Feature engineering
- `capital_net = capital-gain - capital-loss`
- `hours_age_ratio = hours-per-week / (age + 1)`
- `capital_gain_flag`
- `age_band`
- The redundant text `education` feature is removed because `education-num` already provides an ordered numeric representation.

### Leakage prevention
- Train/test split occurs before the pipeline is fitted.
- Imputation/scaling/encoding are fitted only on training data.
- Feature engineering never reads the target.
- Feature importance is calculated from training data.

### Interpretation
Because the Adult dataset contains demographic variables, importance values should be treated as model diagnostics. They do not establish causation, fairness, or an appropriate policy for individuals.


## 12. Final checklist against Task 3

| Requirement | Implementation |
|---|---|
| Complex tabular dataset | UCI Adult: mixed numerical + categorical data |
| `ColumnTransformer` | Yes |
| Numerical scaling | `StandardScaler` |
| Categorical encoding | `OneHotEncoder(handle_unknown="ignore")` |
| Train/test split first | Yes |
| Missing-value handling | Pipeline imputers |
| Feature engineering | Four derived features inside custom transformer |
| Reusable sklearn pipeline | Yes |
| Feature importance | Random Forest tree-based ranking |
| Leakage check | Explicit assertions + pipeline design |
| Expected proof | This Jupyter notebook (`.ipynb`) |

**Submission artifact:** `03_feature_engineering_pipeline.ipynb`
